In [ ]:
# 1. Install required helper libraries
!pip install -q emoji contractions

import re
import emoji
import contractions
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import pos_tag

# Download required NLTK datasets
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

# Hardcoded Corpus
corpus = [
    "I can't wait for the new season of my favorite show! 😍",
    "The COVID-19 pandemic has affected millions of people worldwide.",
    "U.S. stocks fell on Friday after news of rising inflation.",
    "<html><body>Welcome to the website!</body></html>",
    "Python is a great programming language!!! ??",
    "Check out https://www.example.com for more info!",
    "He won 1st prize in the comp3tition!!!",
    "I luvv this movie sooo much!!!"
]


def preprocess_document(text: str) -> dict:
    """Runs the 6 defined NLP techniques + POS Tagging on a text string."""

    # 0. Contraction Expansion (e.g., "can't" -> "cannot")
    expanded_text = contractions.fix(text)

    # 1. Filtration (Removes HTML tags, URLs, emojis, numbers, extra spaces)
    no_html = re.sub(r'<.*?>', '', expanded_text)
    no_urls = re.sub(r'https?://\S+|www\.\S+', '', no_html)
    no_emojis = emoji.replace_emoji(no_urls, replace='')
    no_numbers = re.sub(r'\d+', '', no_emojis)
    filtered_text = re.sub(r'\s+', ' ', no_numbers).strip()

    # 2. Script Validation (Removes punctuation & non-alphabetic special characters)
    script_validated = re.sub(r'[^a-zA-Z\s]', '', filtered_text)
    script_validated = re.sub(r'\s+', ' ', script_validated).strip()
    lowercased_text = script_validated.lower()

    # 3. Tokenization (Splits text into tokens)
    tokens = word_tokenize(lowercased_text)

    # 4. Stopword Removal (Removes common semantic-free words)
    stop_words = set(stopwords.words('english'))
    filtered_tokens = [word for word in tokens if word not in stop_words]

    # 5. Stemming (Reduces words to root form)
    stemmer = PorterStemmer()
    stemmed_tokens = [stemmer.stem(word) for word in filtered_tokens]

    # 6. Lemmatization (Converts words to valid dictionary base form)
    lemmatizer = WordNetLemmatizer()
    lemmatized_tokens = [lemmatizer.lemmatize(word) for word in filtered_tokens]

    # 7. POS Tagging (Assigns grammatical tags)
    pos_tags = pos_tag(filtered_tokens)

    return {
        "filtration": filtered_text,
        "script_validation": script_validated,
        "tokens": tokens,
        "no_stopwords": filtered_tokens,
        "stemmed": stemmed_tokens,
        "lemmatized": lemmatized_tokens,
        "pos_tags": pos_tags
    }


# ==============================================================================
# PIPELINE EXECUTION
# ==============================================================================
print("=" * 80)
print("📊 CORPUS PREPROCESSING RESULTS")
print("=" * 80)

for idx, document in enumerate(corpus, 1):
    results = preprocess_document(document)

    print(f"\n--- [Document {idx}] ---")
    print(f"📥 Original Text:       {document}")
    print(f"1️⃣  Filtration:         {results['filtration']}")
    print(f"2️⃣  Script Validation:  {results['script_validation']}")
    print(f"3️⃣  Tokenization:       {results['tokens']}")
    print(f"4️⃣  Stopwords Removed:  {results['no_stopwords']}")
    print(f"5️⃣  Stemmed Tokens:     {results['stemmed']}")
    print(f"6️⃣  Lemmatized Tokens:  {results['lemmatized']}")
    print(f"7️⃣  POS Tags:           {results['pos_tags']}")
    print("-" * 80)

📊 CORPUS PREPROCESSING RESULTS

--- [Document 1] ---
📥 Original Text:       I can't wait for the new season of my favorite show! 😍
1️⃣  Filtration:         I cannot wait for the new season of my favorite show!
2️⃣  Script Validation:  I cannot wait for the new season of my favorite show
3️⃣  Tokenization:       ['i', 'can', 'not', 'wait', 'for', 'the', 'new', 'season', 'of', 'my', 'favorite', 'show']
4️⃣  Stopwords Removed:  ['wait', 'new', 'season', 'favorite', 'show']
5️⃣  Stemmed Tokens:     ['wait', 'new', 'season', 'favorit', 'show']
6️⃣  Lemmatized Tokens:  ['wait', 'new', 'season', 'favorite', 'show']
7️⃣  POS Tags:           [('wait', 'VB'), ('new', 'JJ'), ('season', 'NN'), ('favorite', 'NN'), ('show', 'NN')]
--------------------------------------------------------------------------------

--- [Document 2] ---
📥 Original Text:       The COVID-19 pandemic has affected millions of people worldwide.
1️⃣  Filtration:         The COVID- pandemic has affected millions of people world

In [ ]:
# Install the lightweight helper library for number verbalization
!pip install -q num2words

import re
import nltk
from num2words import num2words
from nltk.tokenize import word_tokenize
from nltk import pos_tag, ne_chunk
from nltk.tree import Tree
from nltk.util import ngrams

# Download required NLTK resources
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('maxent_ne_chunker', quiet=True)
nltk.download('maxent_ne_chunker_tab', quiet=True)
nltk.download('words', quiet=True)

# Original Corpus
corpus = [
    "Check out https://www.example.com for more info!",
    "He won 1st prize in the comp3tition!!!",
    "I luvv this movie sooo much!!!"
]


# ==============================================================================
# NEW PREPROCESSING FUNCTIONS
# ==============================================================================

def reduce_repeated_characters(text: str) -> str:
    """Technique 1: Reduces 3 or more repeated characters to a single character."""
    pattern = re.compile(r"(.)\1{2,}")
    return pattern.sub(r"\1", text)


def convert_numbers_to_words(text: str) -> str:
    """Technique 2: Converts cardinal numbers and ordinals into spelled-out words."""
    # Convert ordinals (e.g., 1st -> first, 2nd -> second)
    def replace_ordinal(match):
        num = match.group(1)
        try:
            return num2words(int(num), to='ordinal')
        except ValueError:
            return match.group(0)

    # Convert cardinal digits (e.g., 19 -> nineteen)
    def replace_cardinal(match):
        num = match.group(0)
        try:
            return num2words(int(num))
        except ValueError:
            return match.group(0)

    # Process ordinals first, then standard digits
    text = re.sub(r'\b(\d+)(st|nd|rd|th)\b', replace_ordinal, text, flags=re.IGNORECASE)
    text = re.sub(r'\b\d+\b', replace_cardinal, text)
    return text


def mask_named_entities(text: str) -> str:
    """Technique 3: Masks Named Entities (GPE, Person, Organization) with generic tags."""
    tokens = word_tokenize(text)
    tags = pos_tag(tokens)
    chunks = ne_chunk(tags)

    masked_tokens = []
    for chunk in chunks:
        if isinstance(chunk, Tree):
            entity_label = chunk.label()  # e.g., 'GPE', 'ORGANIZATION', 'PERSON'
            masked_tokens.append(f"<{entity_label.upper()}>")
        else:
            masked_tokens.append(chunk[0])

    return " ".join(masked_tokens)


def extract_bigrams(text: str) -> list:
    """Technique 4: Tokenizes text and generates Bigram (2-gram) word pairs."""
    tokens = word_tokenize(text)
    # Filter out pure punctuation tokens for cleaner bigrams
    clean_tokens = [w for w in tokens if w.isalnum()]
    return list(ngrams(clean_tokens, 2))


# ==============================================================================
# EXECUTION ON CORPUS
# ==============================================================================
print("=" * 85)
print("🚀 DEMONSTRATING 4 NEW PREPROCESSING TECHNIQUES")
print("=" * 85)

for idx, doc in enumerate(corpus, 1):
    # Step 1: Character De-elongation
    de_elongated = reduce_repeated_characters(doc)

    # Step 2: Number Verbalization
    verbalized = convert_numbers_to_words(de_elongated)

    # Step 3: Named Entity Masking
    entity_masked = mask_named_entities(verbalized)

    # Step 4: Bigram Generation
    bigrams_list = extract_bigrams(verbalized)

    print(f"\n--- [Document {idx}] ---")
    print(f"📥 Original Text:         {doc}")
    print(f"1️⃣  De-elongated Chars:   {de_elongated}")
    print(f"2️⃣  Verbalized Numbers:   {verbalized}")
    print(f"3️⃣  Masked Entities:      {entity_masked}")
    print(f"4️⃣  Extracted Bigrams:    {bigrams_list}")
    print("-" * 85)

🚀 DEMONSTRATING 4 NEW PREPROCESSING TECHNIQUES

--- [Document 1] ---
📥 Original Text:         Check out https://www.example.com for more info!
1️⃣  De-elongated Chars:   Check out https://w.example.com for more info!
2️⃣  Verbalized Numbers:   Check out https://w.example.com for more info!
3️⃣  Masked Entities:      Check out https : //w.example.com for more info !
4️⃣  Extracted Bigrams:    [('Check', 'out'), ('out', 'https'), ('https', 'for'), ('for', 'more'), ('more', 'info')]
-------------------------------------------------------------------------------------

--- [Document 2] ---
📥 Original Text:         He won 1st prize in the comp3tition!!!
1️⃣  De-elongated Chars:   He won 1st prize in the comp3tition!
2️⃣  Verbalized Numbers:   He won first prize in the comp3tition!
3️⃣  Masked Entities:      He won first prize in the comp3tition !
4️⃣  Extracted Bigrams:    [('He', 'won'), ('won', 'first'), ('first', 'prize'), ('prize', 'in'), ('in', 'the'), ('the', 'comp3tition')]
---------

In [ ]:
class PorterStemmer:
    def __init__(self):
        self.vowels = ('a', 'e', 'i', 'o', 'u')
        self.word = ''
        self.end = 0
        self.start = 0
        self.offset = 0
    def is_vowel(self, letter):
        return letter in self.vowels
    def is_consonant(self, index):
        if self.is_vowel(self.word[index]):
            return False
        if self.word[index] == 'y':
            if index == self.start:
                return True
            else:
                return not self.is_consonant(index - 1)
        return True
    def m(self):
        n = 0
        i = self.start
        while True:
            if i > self.offset:
                return n
            if not self.is_consonant(i):
                break
            i += 1
        i += 1
        while True:
            while True:
                if i > self.offset:
                    return n
                if self.is_consonant(i):
                    break
                i += 1
            i += 1
            n += 1
            while True:
                if i > self.offset:
                    return n
                if not self.is_consonant(i):
                    break
                i += 1
            i += 1
    def contains_vowel(self):
        for i in range(self.start, self.offset + 1):
            if not self.is_consonant(i):
                return True
        return False
    def contains_double_consonant(self, j):
        if j < (self.start + 1):
            return False
        if self.word[j] != self.word[j - 1]:
            return False
        return self.is_consonant(j)
    def is_of_form_cvc(self, i):
        if i < (self.start + 2) or not self.is_consonant(i) or self.is_consonant(i - 1) or not self.is_consonant(i - 2):
            return 0
        ch = self.word[i]
        if ch == 'w' or ch == 'x' or ch == 'y':
            return 0
        return 1
    def ends_with(self, s):
        length = len(s)
        if s[length - 1] != self.word[self.end]:  # tiny speed-up
            return False
        if length > (self.end - self.start + 1):
            return False
        if self.word[self.end - length + 1: self.end + 1] != s:
            return False
        self.offset = self.end - length
        return True
    def set_to(self, s):
        length = len(s)
        self.word = self.word[:self.offset + 1] + s + self.word[self.offset + length + 1:]
        self.end = self.offset + length
    def replace_morpheme(self, s):
        if self.m() > 0:
            self.set_to(s)
    def remove_plurals(self):
        if self.word[self.end] == 's':
            if self.ends_with("sses"):
                self.end = self.end - 2
            elif self.ends_with("ies"):
                self.set_to("y")
            elif self.word[self.end - 1] != 's':
                self.end = self.end - 1
        if self.ends_with("eed"):
            if self.m() > 0:
                self.end = self.end - 1
        elif (self.ends_with("ed") or self.ends_with("ing")) and self.contains_vowel():
            self.end = self.offset
            if self.ends_with("at"):
                self.set_to("ate")
            elif self.ends_with("bl"):
                self.set_to("ble")
            elif self.ends_with("iz"):
                self.set_to("ize")
            elif self.contains_double_consonant(self.end):
                self.end = self.end - 1
                ch = self.word[self.end]
                if ch == 'l' or ch == 's' or ch == 'z':
                    self.end = self.end + 1
            elif self.m() == 1 and self.is_of_form_cvc(self.end):
                self.set_to("e")
    def terminal_y_to_i(self):
        if self.ends_with('y') and self.contains_vowel():
            self.word = self.word[:self.end] + 'i' + self.word[self.end + 1:]
    def map_double_to_single_suffix(self):
        if self.word[self.end - 1] == 'a':
            if self.ends_with("ational"):
                self.replace_morpheme("ate")
            elif self.ends_with("tional"):
                self.replace_morpheme("tion")
        elif self.word[self.end - 1] == 'c':
            if self.ends_with("enci"):
                self.replace_morpheme("ence")
            elif self.ends_with("anci"):
                self.replace_morpheme("ance")
        elif self.word[self.end - 1] == 'e':
            if self.ends_with("izer"):      self.replace_morpheme("ize")
        elif self.word[self.end - 1] == 'l':
            if self.ends_with("bli"):
                self.replace_morpheme("ble")  # --DEPARTURE--
            elif self.ends_with("alli"):
                self.replace_morpheme("al")
            elif self.ends_with("entli"):
                self.replace_morpheme("ent")
            elif self.ends_with("eli"):
                self.replace_morpheme("e")
            elif self.ends_with("ousli"):
                self.replace_morpheme("ous")
        elif self.word[self.end - 1] == 'o':
            if self.ends_with("ization"):
                self.replace_morpheme("ize")
            elif self.ends_with("ation"):
                self.replace_morpheme("ate")
            elif self.ends_with("ator"):
                self.replace_morpheme("ate")
        elif self.word[self.end - 1] == 's':
            if self.ends_with("alism"):
                self.replace_morpheme("al")
            elif self.ends_with("iveness"):
                self.replace_morpheme("ive")
            elif self.ends_with("fulness"):
                self.replace_morpheme("ful")
            elif self.ends_with("ousness"):
                self.replace_morpheme("ous")
        elif self.word[self.end - 1] == 't':
            if self.ends_with("aliti"):
                self.replace_morpheme("al")
            elif self.ends_with("iviti"):
                self.replace_morpheme("ive")
            elif self.ends_with("biliti"):
                self.replace_morpheme("ble")
        elif self.word[self.end - 1] == 'g':
            if self.ends_with("logi"):      self.replace_morpheme("log")
    def step3(self):
        if self.word[self.end] == 'e':
            if self.ends_with("icate"):
                self.replace_morpheme("ic")
            elif self.ends_with("ative"):
                self.replace_morpheme("")
            elif self.ends_with("alize"):
                self.replace_morpheme("al")
        elif self.word[self.end] == 'i':
            if self.ends_with("iciti"):     self.replace_morpheme("ic")
        elif self.word[self.end] == 'l':
            if self.ends_with("ical"):
                self.replace_morpheme("ic")
            elif self.ends_with("ful"):
                self.replace_morpheme("")
        elif self.word[self.end] == 's':
            if self.ends_with("ness"):      self.replace_morpheme("")
    def step4(self):
        if self.word[self.end - 1] == 'a':
            if self.ends_with("al"):
                pass
            else:
                return
        elif self.word[self.end - 1] == 'c':
            if self.ends_with("ance"):
                pass
            elif self.ends_with("ence"):
                pass
            else:
                return
        elif self.word[self.end - 1] == 'e':
            if self.ends_with("er"):
                pass
            else:
                return
        elif self.word[self.end - 1] == 'i':
            if self.ends_with("ic"):
                pass
            else:
                return
        elif self.word[self.end - 1] == 'l':
            if self.ends_with("able"):
                pass
            elif self.ends_with("ible"):
                pass
            else:
                return
        elif self.word[self.end - 1] == 'n':
            if self.ends_with("ant"):
                pass
            elif self.ends_with("ement"):
                pass
            elif self.ends_with("ment"):
                pass
            elif self.ends_with("ent"):
                pass
            else:
                return
        elif self.word[self.end - 1] == 'o':
            if self.ends_with("ion") and (self.word[self.offset] == 's' or self.word[self.offset] == 't'):
                pass
            elif self.ends_with("ou"):
                pass
            else:
                return
        elif self.word[self.end - 1] == 's':
            if self.ends_with("ism"):
                pass
            else:
                return
        elif self.word[self.end - 1] == 't':
            if self.ends_with("ate"):
                pass
            elif self.ends_with("iti"):
                pass
            else:
                return
        elif self.word[self.end - 1] == 'u':
            if self.ends_with("ous"):
                pass
            else:
                return
        elif self.word[self.end - 1] == 'v':
            if self.ends_with("ive"):
                pass
            else:
                return
        elif self.word[self.end - 1] == 'z':
            if self.ends_with("ize"):
                pass
            else:
                return
        else:
            return
        if self.m() > 1:
            self.end = self.offset
    def step5(self):
        self.offset = self.end
        if self.word[self.end] == 'e':
            a = self.m()
            if a > 1 or (a == 1 and not self.is_of_form_cvc(self.end - 1)):
                self.end = self.end - 1
        if self.word[self.end] == 'l' and self.contains_double_consonant(self.end) and self.m() > 1:
            self.end = self.end - 1
    def stem_document(self, document):
        result = []
        for line in document.split('\n'):
            result.append(self.stem_sentence(line))
        return '\n'.join(result)
    def alphabetic(self, word):
        return ''.join([letter if letter.isalpha() else '' for letter in word])
    def stem_sentence(self, sentence):
        result = []
        for word in sentence.split():
            result.append(self.stem_word(word))
        return ' '.join(result)
    def stem_word(self, word):
        if word == '':
            return ''
        self.word = word
        self.end = len(word) - 1
        self.start = 0
        self.remove_plurals()
        self.terminal_y_to_i()
        self.map_double_to_single_suffix()
        self.step3()
        self.step4()
        self.step5()
        return self.word[self.start: self.end + 1]
def display_menu():
    print("\n" + "="*50)
    print("           PORTER STEMMER")
    print("="*50)
    print("1. Stem a single word")
    print("2. Run demo with test words")
    print("3. Exit")
    print("="*50)
def run_demo():
    stemmer = PorterStemmer()
    test_words = ["caresses", "ponies", "mating", "meeting", "happiness", "relational", "skies"]
    print("Stemmed words:")
    for word in test_words:
        stemmed = stemmer.stem_word(word)
        print(f"{word} → {stemmed}")
def stem_single_word():
    stemmer = PorterStemmer()
    while True:
        print("\n" + "-"*40)
        word = input("Enter a word to stem (or 'back' to return): ").strip()
        if word.lower() == 'back':
            break
        if not word:
            print("Please enter a valid word!")
            continue
        clean_word = stemmer.alphabetic(word.lower())
        if not clean_word:
            print("Please enter a word with alphabetic characters!")
            continue
        stemmed = stemmer.stem_word(clean_word)
        print(f"\nOriginal: '{word}'")
        print(f"Cleaned:  '{clean_word}'")
        print(f"Stemmed:  '{stemmed}'")
def main():
    print("Welcome to the Porter Stemmer!")
    while True:
        display_menu()
        try:
            choice = input("\nEnter your choice (1-3): ").strip()
            if choice == '1':
                stem_single_word()
            elif choice == '2':
                run_demo()
            elif choice == '3':
                print("\nThank you for using Porter Stemmer! Goodbye!")
                break
            else:
                print("\nInvalid choice! Please enter a number between 1-6.")
        except KeyboardInterrupt:
            print("\n\nProgram interrupted. Goodbye!")
            break
        except Exception as e:
            print(f"\nAn error occurred: {e}")
            print("Please try again.")
if __name__ == "__main__":
    main()

Welcome to the Porter Stemmer!

           PORTER STEMMER
1. Stem a single word
2. Run demo with test words
3. Exit

Invalid choice! Please enter a number between 1-6.

           PORTER STEMMER
1. Stem a single word
2. Run demo with test words
3. Exit

----------------------------------------

Original: 'skies'
Cleaned:  'skies'
Stemmed:  'sky'

----------------------------------------

Original: 'berries'
Cleaned:  'berries'
Stemmed:  'berri'

----------------------------------------

Original: 'lies'
Cleaned:  'lies'
Stemmed:  'ly'

----------------------------------------

Original: 'flies'
Cleaned:  'flies'
Stemmed:  'fly'

----------------------------------------

Original: 'educational'
Cleaned:  'educational'
Stemmed:  'educ'

----------------------------------------
